In [ ]:
# Bước 1: dựng bảng sức chứa
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

f1 = lambda a, b: f1_score(a, b, average="macro", labels=[0, 1], zero_division=0)
Xh, Xk, yh, yk = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

for ten, C, ng, mdf in (("nho", 0.1, (2, 3), 5),
                         ("vua", 4.0, (2, 5), 2),
                         ("lon", 1000.0, (1, 8), 1)):
    m = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=ng, min_df=mdf),
        LogisticRegression(max_iter=4000, C=C, random_state=0))
    m.fit(Xh, yh)
    a, b = f1(yh, m.predict(Xh)), f1(yk, m.predict(Xk))
    so_dt = len(m.named_steps["tfidfvectorizer"].vocabulary_)
    print("%-4s %5d dac trung  hoc %.4f  kiem %.4f  chenh %.4f"
          % (ten, so_dt, a, b, a - b))


In [ ]:
#Bước 2: xác thực chéo

import numpy as np
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score

cham = make_scorer(f1_score, average="macro", labels=[0, 1], zero_division=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
d = cross_val_score(mo_hinh(), X, y, cv=cv, scoring=cham)
print("nam diem:", np.round(d, 4))
print("trung binh %.4f, do lech chuan %.4f" % (d.mean(), d.std()))


In [ ]:
#Bước 3: đo độ ổn định của chính cách đo
mot_lan, nhieu_lan = [], []
for s in (0, 1, 2, 42):
    Xh, Xk, yh, yk = train_test_split(X, y, test_size=0.2, random_state=s, stratify=y)
    m = mo_hinh()
    m.fit(Xh, yh)
    mot_lan.append(f1(yk, m.predict(Xk)))

    cvs = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
    nhieu_lan.append(cross_val_score(mo_hinh(), X, y, cv=cvs, scoring=cham).mean())

print("mot lan chia: dai %.4f" % (max(mot_lan) - min(mot_lan)))
print("CV 5 phan   : dai %.4f" % (max(nhieu_lan) - min(nhieu_lan)))
